In [ ]:
import random, math, csv
import numpy as np
from typing import List
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm
import plotly.graph_objects as go
from torch.nn.utils import spectral_norm


# ============================================================
# CONFIG
# ============================================================
SEED        = 42
EPOCHS      = 80
BATCH_SIZE  = 128

LR           = 1e-3
WEIGHT_DECAY = 1e-4

ALPHA_CE     = 1.0    # CE weight
HINGE_WEIGHT = 1.0    # margin hinge weight
RECON_WEIGHT = 0.1    # reconstruction loss weight (reduced to help classification)

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0
PERSISTENT  = False
PIN_MEMORY  = (DEVICE == "cuda")

MARGIN        = 1.0
SIGMA_Q       = 0.4       # example shift level for printing
ALPHA         = 0.2       # nominal miscoverage (target coverage 1-ALPHA)
CLS_SCALE     = 1.0       # global scale on logits

# σ-grid for coverage / set-size surfaces
SIGMA_MIN_GRID = 0.0
SIGMA_MAX_GRID = 1.5
N_SIGMA_GRID   = 11
SIGMA_GRID     = np.linspace(SIGMA_MIN_GRID, SIGMA_MAX_GRID, N_SIGMA_GRID)

NUM_CLASSES = 10
LATENT_DIM  = 128         # latent dimension for CIFAR AE

SMOOTH_WINDOW_EPOCHS = 3  # smoothing for 3D plots

# ============================================================
# SEED
# ============================================================
def set_seed(seed: int = 43):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# ============================================================
# MODEL: Conv Autoencoder + Linear classifier for CIFAR-10
# ============================================================
class AELinearCIFAR10(nn.Module):
    """
    Conv autoencoder + linear classifier for 32x32 RGB images:
      encoder: Conv blocks -> flatten -> FC -> latent
      decoder: latent -> FC -> reshape -> ConvTranspose blocks -> 3x32x32
      classifier: latent -> NUM_CLASSES (single Linear head)
    """
    def __init__(self, latent_dim: int = LATENT_DIM, num_classes: int = NUM_CLASSES):
        super().__init__()

        # Encoder: 32x32 -> 16x16 -> 8x8
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # 16x16
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # 8x8
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.flatten = nn.Flatten()
        self.enc_fc = nn.Sequential(
            nn.Linear(128 * 8 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, latent_dim),
        )

        # Decoder: latent -> 128x8x8 -> 3x32x32
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 128 * 8 * 8),
            nn.ReLU(inplace=True),
        )
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # 8x8 -> 16x16
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),   # 16x16 -> 32x32
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 3, kernel_size=3, stride=1, padding=1),
        )

        # Linear classifier on latent features
        self.classifier = spectral_norm(nn.Linear(latent_dim, num_classes))

    def encode(self, x):
        h = self.encoder_conv(x)
        h = self.flatten(h)
        z = self.enc_fc(h)
        return z

    def decode(self, z):
        h = self.dec_fc(z)
        h = h.view(-1, 128, 8, 8)
        x_rec = self.decoder_conv(h)
        return x_rec

    def forward(self, x):
        z = self.encode(x)
        logits = self.classifier(z)
        x_rec = self.decode(z)
        return CLS_SCALE * logits, x_rec

    @torch.no_grad()
    def logits_from_x_nograd(self, x):
        z = self.encode(x)
        logits = self.classifier(z)
        return CLS_SCALE * logits

    @torch.no_grad()
    def features_from_x_nograd(self, x):
        # latent features φ(x)
        return self.encode(x)

# ============================================================
# LOSSES / METRICS
# ============================================================
def cross_entropy_from_logits(logits, y, label_smoothing: float = 0.0):
    return F.cross_entropy(logits, y, reduction='mean', label_smoothing=label_smoothing)

def margin_hinge_loss_from_logits(logits, targets, margin=1.0):
    idx = torch.arange(logits.size(0), device=logits.device)
    masked = logits.clone()
    masked[idx, targets] = -float('inf')
    max_other = masked.max(dim=1).values
    m = logits[idx, targets] - max_other
    v = F.relu(margin - m)
    return v.mean()

@torch.no_grad()
def samplewise_margin_and_hinge(logits, targets, margin=1.0):
    """
    Return per-sample (margin, hinge) without reduction.
    margin_i = logit_y - max_{k != y} logit_k
    hinge_i  = relu(margin_const - margin_i)
    """
    idx = torch.arange(logits.size(0), device=logits.device)
    masked = logits.clone()
    masked[idx, targets] = -float('inf')
    max_other = masked.max(dim=1).values
    m = logits[idx, targets] - max_other
    h = F.relu(margin - m)
    return m, h

# ============================================================
# DATA & LOADERS (CIFAR-10)
# ============================================================
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

transform_test  = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_set = datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform_train)
test_set  = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

def make_half_splits(dataset, batch_size=128):
    n = len(dataset)
    idx_cal = np.arange(0, n//2)
    idx_tst = np.arange(n//2, n)
    ds_cal  = Subset(dataset, idx_cal)
    ds_tst  = Subset(dataset, idx_tst)
    ld_cal  = DataLoader(ds_cal, batch_size=batch_size, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    ld_tst  = DataLoader(ds_tst, batch_size=batch_size, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    return ld_cal, ld_tst

P_cal_loader, P_tst_loader = make_half_splits(test_set, BATCH_SIZE)
Q_cal_loader, Q_tst_loader = P_cal_loader, P_tst_loader  # Q realized by T_sigma at scoring time

# ============================================================
# Q-SHIFT TRANSFORM T_sigma
# ============================================================
def apply_Q_transform(x: torch.Tensor, sigma: float) -> torch.Tensor:
    """
    Structured shift T_sigma(x) used to define Q for CIFAR-10:
      - brightness/contrast jitter per channel
      - small translations (torch.roll)
      - mild additive Gaussian noise
    Acts in normalized CIFAR space.
    """
    if sigma <= 0.0:
        return x

    B = x.size(0)
    device = x.device

    contrast_scale = 1.0 + 0.8 * sigma * (2.0 * torch.rand(B, 3, 1, 1, device=device) - 1.0)
    brightness_shift = 0.4 * sigma * (2.0 * torch.rand(B, 3, 1, 1, device=device) - 1.0)
    x = x * contrast_scale + brightness_shift

    max_shift = int(4 * sigma)  # up to 4 px at sigma=1
    if max_shift > 0:
        shift_x = torch.randint(-max_shift, max_shift + 1, (B,), device=device)
        shift_y = torch.randint(-max_shift, max_shift + 1, (B,), device=device)
        for i in range(B):
            x[i] = torch.roll(x[i], shifts=(int(shift_y[i]), int(shift_x[i])), dims=(1, 2))

    noise_std = 0.1 * sigma
    if noise_std > 0:
        x = x + noise_std * torch.randn_like(x)

    return x

# ============================================================
# MODEL + OPTIMIZER
# ============================================================
model = AELinearCIFAR10(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = None  # keep simple

# ============================================================
# TRAIN / EVAL
# ============================================================
def train_one_epoch(epoch: int):
    model.train()
    sums = {'cce':0.0,'hinge':0.0,'recon':0.0,'acc':0.0,'n':0}
    pbar = tqdm(DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT),
                desc=f"Train {epoch:02d}")
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits, x_rec = model(x)
        cce   = cross_entropy_from_logits(logits, y, label_smoothing=0.0)
        hinge = margin_hinge_loss_from_logits(logits, y, margin=MARGIN)
        recon = F.mse_loss(x_rec, x)

        loss = ALPHA_CE * cce + HINGE_WEIGHT * hinge + RECON_WEIGHT * recon

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            acc = (logits.argmax(1) == y).float().mean().item()
        bs = x.size(0)
        sums['cce']   += cce.item()*bs
        sums['hinge'] += hinge.item()*bs
        sums['recon'] += recon.item()*bs
        sums['acc']   += acc*bs
        sums['n']     += bs
        pbar.set_postfix({
            'cce':   f"{sums['cce']/sums['n']:.4f}",
            'hinge': f"{sums['hinge']/sums['n']:.4f}",
            'recon': f"{sums['recon']/sums['n']:.4f}",
            'acc':   f"{sums['acc']/sums['n']:.4f}",
        })
    for k in ['cce','hinge','recon','acc']:
        sums[k] /= sums['n']
    return sums

@torch.no_grad()
def evaluate(loader, tag="Eval"):
    model.eval()
    tot = {'cce':0.0,'hinge':0.0,'acc':0.0,'n':0}
    pbar = tqdm(loader, desc=f"{tag}", leave=False)
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        cce   = F.cross_entropy(logits, y, reduction='mean').item()
        hinge = margin_hinge_loss_from_logits(logits, y, margin=MARGIN).item()
        acc   = (logits.argmax(1) == y).float().mean().item()
        bs = x.size(0)
        tot['cce']  += cce*bs
        tot['hinge']+= hinge*bs
        tot['acc']  += acc*bs
        tot['n']    += bs
    for k in ['cce','hinge','acc']:
        tot[k] /= tot['n']
    return tot

# ============================================================
# CONFORMAL HELPERS
# ============================================================
@torch.no_grad()
def predictive_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    probs = F.softmax(logits, dim=1)
    log_probs = (probs + 1e-12).log()
    ent = -(probs * log_probs).sum(dim=1)
    return ent

@torch.no_grad()
def scores_for_labels_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> np.ndarray:
    B = logits.size(0)
    idx = torch.arange(B, device=logits.device)
    masked = logits.clone()
    masked[idx, labels] = -float('inf')
    max_other = masked.max(dim=1).values
    s = (max_other - logits[idx, labels]).detach().float().cpu().numpy()
    return s

def conformal_quantile(scores: np.ndarray, alpha: float, finite_sample: bool = True) -> float:
    n = scores.size
    if n == 0:
        return float('nan')
    if finite_sample:
        qlevel = math.ceil((n + 1) * (1 - alpha)) / n
    else:
        qlevel = 1 - alpha
    try:
        return float(np.quantile(scores, qlevel, method='higher'))
    except TypeError:
        return float(np.quantile(scores, qlevel, interpolation='higher'))

def coverage_given_threshold(scores_true: np.ndarray, threshold: float) -> float:
    if scores_true.size == 0:
        return float('nan')
    return float((scores_true <= threshold).mean())

@torch.no_grad()
def collect_true_scores(loader: DataLoader, sigma: float) -> np.ndarray:
    model.eval()
    all_scores = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        s = scores_for_labels_from_logits(logits, y)
        all_scores.append(s)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

@torch.no_grad()
def expected_set_size_threshold(loader: DataLoader, sigma: float, q: float) -> float:
    model.eval()
    total_size = 0.0
    total_n = 0
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)  # [B,K]
        B, K = logits.shape

        values, indices = logits.topk(k=2, dim=1)
        max1 = values[:, 0:1]
        max2 = values[:, 1:2]
        argmax = indices[:, 0]

        classes = torch.arange(K, device=logits.device).unsqueeze(0).expand(B, -1)
        argmax_exp = argmax.unsqueeze(1).expand(-1, K)
        max_other = torch.where(classes == argmax_exp, max2, max1)

        scores_all = max_other - logits
        set_sizes = (scores_all <= q).sum(dim=1).float()

        total_size += set_sizes.sum().item()
        total_n += B

    return total_size / total_n if total_n > 0 else float('nan')

@torch.no_grad()
def collect_pseudo_scores_mixed(
    loader: DataLoader,
    sigma: float,
    u: float,
    num_classes: int = NUM_CLASSES,
) -> np.ndarray:
    """
    Mixed pseudo-labels: if entropy <= u, use argmax; else random label.
    """
    model.eval()
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        ent = predictive_entropy_from_logits(logits)
        yhat = logits.argmax(dim=1)

        B, K = logits.shape
        rand_labels = torch.randint(low=0, high=num_classes, size=(B,), device=logits.device)
        mask_low = (ent <= u)
        pseudo_y = torch.where(mask_low, yhat, rand_labels)

        s = scores_for_labels_from_logits(logits, pseudo_y)
        all_scores.append(s)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

@torch.no_grad()
def compute_uP_star(
    P_cal_loader: DataLoader,
    P_tst_loader: DataLoader,
    alpha: float = ALPHA,
    num_u_points: int = 30,
):
    ent_all = []
    for x, _ in P_cal_loader:
        x = x.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        ent = predictive_entropy_from_logits(logits)
        ent_all.append(ent.cpu().numpy())
    ent_all = np.concatenate(ent_all, axis=0)

    low_q, high_q = 0.05, 0.95
    t_min = np.quantile(ent_all, low_q)
    t_max = np.quantile(ent_all, high_q)
    u_cutoffs = np.linspace(t_min, t_max, num_u_points)

    scores_P_tst_true = collect_true_scores(P_tst_loader, sigma=0.0)
    cov_P_vals = []
    target_cov = 1.0 - alpha

    u_P_star = u_cutoffs[0]
    q_Phat_uP_star = None

    for u in u_cutoffs:
        scores_P_cal_pseudo = collect_pseudo_scores_mixed(
            loader=P_cal_loader,
            sigma=0.0,
            u=u,
            num_classes=NUM_CLASSES,
        )
        q_Phat_u = conformal_quantile(scores_P_cal_pseudo, alpha, finite_sample=True)
        cov_P_u = coverage_given_threshold(scores_P_tst_true, q_Phat_u)
        cov_P_vals.append(cov_P_u)

        if cov_P_u >= target_cov:
            u_P_star = u
            q_Phat_uP_star = q_Phat_u

    cov_P_vals = np.array(cov_P_vals)

    if q_Phat_uP_star is None:
        scores_P_cal_pseudo = collect_pseudo_scores_mixed(
            loader=P_cal_loader,
            sigma=0.0,
            u=u_cutoffs[-1],
            num_classes=NUM_CLASSES,
        )
        q_Phat_uP_star = conformal_quantile(scores_P_cal_pseudo, alpha, finite_sample=True)
        u_P_star = float(u_cutoffs[-1])

    return float(u_P_star), u_cutoffs, cov_P_vals, float(q_Phat_uP_star)

# ============================================================
# STORAGE FOR PER-EPOCH STATS
# ============================================================
hinge_P_epochs         = np.zeros(EPOCHS, dtype=np.float64)
cov_Q_uP_epochs_sigma  = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)
cov_P_uP_epochs        = np.zeros(EPOCHS, dtype=np.float64)
u_P_star_epochs        = np.zeros(EPOCHS, dtype=np.float64)

ESS_P_uP_epochs        = np.zeros(EPOCHS, dtype=np.float64)
ESS_P_oracle_epochs    = np.zeros(EPOCHS, dtype=np.float64)
ESS_Q_uP_epochs_sigma      = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)
ESS_Q_oracle_epochs_sigma  = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)

# ============================================================
# MAIN TRAINING + COVERAGE / SET-SIZE LOOP
# ============================================================
print("=== Training with per-epoch source-tuned pseudo-label coverage & set-size tracking (Conv AE + Linear, CIFAR-10) ===")

for epoch in range(1, EPOCHS+1):
    tr = train_one_epoch(epoch)
    teP = evaluate(P_tst_loader, tag=f"Test(P) epoch {epoch}")
    hinge_P = teP["hinge"]
    hinge_P_epochs[epoch-1] = hinge_P

    # Compute u_P^* for current model
    u_P_star, u_grid, cov_P_over_u, q_Phat_uP_star = compute_uP_star(
        P_cal_loader=P_cal_loader,
        P_tst_loader=P_tst_loader,
        alpha=ALPHA,
        num_u_points=30,
    )
    u_P_star_epochs[epoch-1] = u_P_star

    scores_P_tst_true = collect_true_scores(P_tst_loader, sigma=0.0)
    cov_P_uP_star = coverage_given_threshold(scores_P_tst_true, q_Phat_uP_star)
    cov_P_uP_epochs[epoch-1] = cov_P_uP_star

    scores_P_cal_true = collect_true_scores(P_cal_loader, sigma=0.0)
    q_P_true_oracle = conformal_quantile(scores_P_cal_true, ALPHA, finite_sample=True)

    ESS_P_uP = expected_set_size_threshold(P_tst_loader, sigma=0.0, q=q_Phat_uP_star)
    ESS_P_oracle = expected_set_size_threshold(P_tst_loader, sigma=0.0, q=q_P_true_oracle)
    ESS_P_uP_epochs[epoch-1] = ESS_P_uP
    ESS_P_oracle_epochs[epoch-1] = ESS_P_oracle

    print(
        f"Epoch {epoch:03d} | "
        f"Train CE {tr['cce']:.4f} hinge {tr['hinge']:.4f} recon {tr['recon']:.4f} acc {tr['acc']:.4f} | "
        f"Test(P) CE {teP['cce']:.4f} hinge {teP['hinge']:.4f} acc {teP['acc']:.4f} | "
        f"u_P^*={u_P_star:.4f}, Cov_P(u_P^*)={cov_P_uP_star:.4f}, "
        f"ESS_P(u_P^*)={ESS_P_uP:.2f}, ESS_P(oracle)={ESS_P_oracle:.2f}"
    )

    if scheduler is not None:
        scheduler.step()

    # For each sigma, compute coverage and ESS on Q
    for s_idx, sigma in enumerate(SIGMA_GRID):
        sigma_val = float(sigma)

        scores_Q_cal_pseudo = collect_pseudo_scores_mixed(
            loader=Q_cal_loader,
            sigma=sigma_val,
            u=u_P_star,
            num_classes=NUM_CLASSES,
        )
        q_Qhat_uP_sigma = conformal_quantile(scores_Q_cal_pseudo, ALPHA, finite_sample=True)

        scores_Q_cal_true = collect_true_scores(Q_cal_loader, sigma=sigma_val)
        q_Q_true_sigma = conformal_quantile(scores_Q_cal_true, ALPHA, finite_sample=True)

        scores_Q_tst_true = collect_true_scores(
            loader=Q_tst_loader,
            sigma=sigma_val,
        )
        cov_Q_uP = coverage_given_threshold(scores_Q_tst_true, q_Qhat_uP_sigma)
        cov_Q_uP_epochs_sigma[epoch-1, s_idx] = cov_Q_uP

        ESS_Q_uP = expected_set_size_threshold(Q_tst_loader, sigma=sigma_val, q=q_Qhat_uP_sigma)
        ESS_Q_oracle = expected_set_size_threshold(Q_tst_loader, sigma=sigma_val, q=q_Q_true_sigma)
        ESS_Q_uP_epochs_sigma[epoch-1, s_idx] = ESS_Q_uP
        ESS_Q_oracle_epochs_sigma[epoch-1, s_idx] = ESS_Q_oracle

    if SIGMA_MIN_GRID <= SIGMA_Q <= SIGMA_MAX_GRID:
        s_idx_print = int(
            round(
                (SIGMA_Q - SIGMA_MIN_GRID)
                / (SIGMA_MAX_GRID - SIGMA_MIN_GRID)
                * (N_SIGMA_GRID - 1)
            )
        )
        cov_Q_at_SIGMAQ = cov_Q_uP_epochs_sigma[epoch-1, s_idx_print]
        ESS_Q_uP_at_SIGMAQ = ESS_Q_uP_epochs_sigma[epoch-1, s_idx_print]
        ESS_Q_oracle_at_SIGMAQ = ESS_Q_oracle_epochs_sigma[epoch-1, s_idx_print]
        print(
            f"  Q at σ={SIGMA_Q:.2f} using u_P^*: "
            f"Coverage={cov_Q_at_SIGMAQ:.4f}, "
            f"ESS_Q(u_P^*)={ESS_Q_uP_at_SIGMAQ:.2f}, "
            f"ESS_Q(oracle)={ESS_Q_oracle_at_SIGMAQ:.2f}"
        )

# ============================================================
# SMOOTHING + 3D PLOTS + CSV (like MNIST)
# ============================================================
def smooth_1d(a: np.ndarray, window: int = 3) -> np.ndarray:
    a = np.asarray(a, dtype=np.float64)
    if window <= 1 or a.size == 0 or a.size < window:
        return a.copy()
    pad_left = window // 2
    pad_right = window - 1 - pad_left
    a_pad = np.pad(a, (pad_left, pad_right), mode="edge")
    kernel = np.ones(window, dtype=np.float64) / float(window)
    return np.convolve(a_pad, kernel, mode="valid")

def smooth_2d_along_epochs(Z: np.ndarray, window: int = 3) -> np.ndarray:
    Z = np.asarray(Z, dtype=np.float64)
    if window <= 1 or Z.size == 0 or Z.shape[0] < window:
        return Z.copy()
    Z_sm = np.empty_like(Z)
    for j in range(Z.shape[1]):
        Z_sm[:, j] = smooth_1d(Z[:, j], window)
    return Z_sm

hinge_P_plot     = smooth_1d(hinge_P_epochs, window=SMOOTH_WINDOW_EPOCHS)
u_P_star_plot    = smooth_1d(u_P_star_epochs, window=SMOOTH_WINDOW_EPOCHS)
cov_Q_uP_plot    = smooth_2d_along_epochs(cov_Q_uP_epochs_sigma, window=SMOOTH_WINDOW_EPOCHS)
ESS_Q_uP_plot    = smooth_2d_along_epochs(ESS_Q_uP_epochs_sigma, window=SMOOTH_WINDOW_EPOCHS)

print("Smoothing done for plotting (window =", SMOOTH_WINDOW_EPOCHS, ")")

LOSS_AXIS  = hinge_P_plot
SIGMA_AXIS = SIGMA_GRID
SIGMA_MESH, LOSS_MESH = np.meshgrid(SIGMA_AXIS, LOSS_AXIS, indexing="ij")
Z_cov = cov_Q_uP_plot.T

nominal_cov = 1.0 - ALPHA
Z_nom = nominal_cov * np.ones_like(Z_cov)

fig3d_cov = go.Figure()
fig3d_cov.add_surface(
    x=SIGMA_MESH, y=LOSS_MESH, z=Z_cov,
    colorscale="Viridis", opacity=0.9,
    name="Coverage on Q using u_P^*", showscale=True,
)
fig3d_cov.add_surface(
    x=SIGMA_MESH, y=LOSS_MESH, z=Z_nom,
    colorscale=[[0, "lightgrey"], [1, "lightgrey"]],
    opacity=0.3, name="Nominal 1−α plane", showscale=False,
)
fig3d_cov.update_layout(
    title="CIFAR-10 (Conv AE + linear): Coverage on Q vs (σ, hinge loss on P)\nSource-tuned mixed pseudo-labeling (u_P^*)",
    scene=dict(
        xaxis_title="σ (Q shift level)",
        yaxis_title="Hinge loss on P (P_tst)",
        zaxis_title="Coverage on Q",
        zaxis=dict(range=[0.0, 1.02]),
    ),
    width=900, height=700,
)
fig3d_cov.write_html("cifar10_ae_linear_uP_coverage_vs_sigma_hinge_3d.html", include_plotlyjs="cdn")
print("Saved 3D coverage plot to cifar10_ae_linear_uP_coverage_vs_sigma_hinge_3d.html")

Z_ess_Q_uP = ESS_Q_uP_plot.T
fig3d_ess = go.Figure()
fig3d_ess.add_surface(
    x=SIGMA_MESH, y=LOSS_MESH, z=Z_ess_Q_uP,
    colorscale="Viridis", opacity=0.9,
    name="Expected set size on Q using u_P^*", showscale=True,
)
fig3d_ess.update_layout(
    title="CIFAR-10 (Conv AE + linear): Expected set size on Q vs (σ, hinge loss on P)\nSource-tuned mixed pseudo-labeling (u_P^*)",
    scene=dict(
        xaxis_title="σ (Q shift level)",
        yaxis_title="Hinge loss on P (P_tst)",
        zaxis_title="Expected set size on Q",
        zaxis=dict(range=[0.5, NUM_CLASSES + 0.5]),
    ),
    width=900, height=700,
)
fig3d_ess.write_html("cifar10_ae_linear_uP_exp_set_size_vs_sigma_hinge_3d.html", include_plotlyjs="cdn")
print("Saved 3D expected set size plot to cifar10_ae_linear_uP_exp_set_size_vs_sigma_hinge_3d.html")

fig_u_vs_hinge = go.Figure()
fig_u_vs_hinge.add_trace(
    go.Scatter(
        x=hinge_P_plot,
        y=u_P_star_plot,
        mode="markers+lines",
        name="u_P^* vs hinge(P)"
    )
)
fig_u_vs_hinge.update_layout(
    title="CIFAR-10 (Conv AE + linear): u_P^* vs hinge loss on P_tst (smoothed)",
    xaxis_title="Hinge loss on P_tst",
    yaxis_title="u_P^* (entropy cutoff)",
    width=800, height=500,
)
fig_u_vs_hinge.write_html("cifar10_ae_linear_uP_vs_hingeP.html", include_plotlyjs="cdn")
print("Saved u_P^* vs hinge(P) plot to cifar10_ae_linear_uP_vs_hingeP.html")

csv_path = "cifar10_ae_linear_uP_expected_set_sizes.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "domain", "sigma", "hinge_P",
                     "ESS_uP", "ESS_oracle"])
    for epoch in range(1, EPOCHS+1):
        hinge = hinge_P_epochs[epoch-1]
        writer.writerow([
            epoch, "P", 0.0, hinge,
            ESS_P_uP_epochs[epoch-1],
            ESS_P_oracle_epochs[epoch-1],
        ])
        for s_idx, sigma in enumerate(SIGMA_GRID):
            writer.writerow([
                epoch, "Q", float(sigma), hinge,
                ESS_Q_uP_epochs_sigma[epoch-1, s_idx],
                ESS_Q_oracle_epochs_sigma[epoch-1, s_idx],
            ])

print(f"Saved expected set size table to {csv_path}")

print("\nDone. Arrays available (raw, unsmoothed):")
print("hinge_P_epochs.shape =", hinge_P_epochs.shape)
print("cov_Q_uP_epochs_sigma.shape =", cov_Q_uP_epochs_sigma.shape)
print("cov_P_uP_epochs.shape =", cov_P_uP_epochs.shape)
print("u_P_star_epochs.shape =", u_P_star_epochs.shape)
print("ESS_P_uP_epochs.shape =", ESS_P_uP_epochs.shape)
print("ESS_P_oracle_epochs.shape =", ESS_P_oracle_epochs.shape)
print("ESS_Q_uP_epochs_sigma.shape =", ESS_Q_uP_epochs_sigma.shape)
print("ESS_Q_oracle_epochs_sigma.shape =", ESS_Q_oracle_epochs_sigma.shape)

# ============================================================
# SECOND STAGE: HARD PSEUDO-CALIBRATION + COVERAGE LBs (CIFAR-10)
# ============================================================
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import scienceplots
import matplotlib as mpl

plt.style.use(["science", "ieee", "no-latex"])

mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["axes.titlesize"] = 12
mpl.rcParams["legend.fontsize"] = 9
mpl.rcParams["xtick.labelsize"] = 11
mpl.rcParams["ytick.labelsize"] = 11
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

IEEE_WIDTH_1COL = 5.0
IEEE_ASPECT = 0.9
FIGSIZE_1COL = (IEEE_WIDTH_1COL, IEEE_WIDTH_1COL * IEEE_ASPECT)

# -----------------------------------
# Helper: hard pseudo-calibration scores on Q
# -----------------------------------
@torch.no_grad()
def collect_pseudo_scores_hard(
    loader: DataLoader,
    sigma: float,
    num_classes: int = NUM_CLASSES,
) -> np.ndarray:
    """
    Hard pseudo-calibration:
      pseudo-label = argmax_k logit_k for all x (after applying T_sigma).
    """
    model.eval()
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        yhat = logits.argmax(dim=1)
        s = scores_for_labels_from_logits(logits, yhat)
        all_scores.append(s)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

# -----------------------------------
# Helper: latent features ϕ(x) = encoder output
# -----------------------------------
@torch.no_grad()
def extract_features_batch(x: torch.Tensor) -> torch.Tensor:
    model.eval()
    return model.features_from_x_nograd(x)

# -----------------------------------
# Helper: latent ρ_mix(σ) with trimming
# -----------------------------------
@torch.no_grad()
def estimate_feature_rho_mix(
    loader: DataLoader,
    sigma: float,
    max_batches: int = None,
    trim_quantile: float = 0.9,
):
    """
    Latent-space shift magnitude:

        ρ_mix(σ) ≈ E[ ||ϕ(X) - ϕ(T_σ(X))||_2 ].

    Returns (rho_full, rho_trimmed).
    """
    model.eval()
    all_dists = []

    for b, (x, _) in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break

        x = x.to(DEVICE)
        z = extract_features_batch(x)
        x_shift = apply_Q_transform(x.clone(), sigma)
        z_shift = extract_features_batch(x_shift)

        diff = z - z_shift
        d = diff.norm(p=2, dim=1)
        all_dists.append(d.detach().cpu().numpy())

    if not all_dists:
        return float("nan"), float("nan")

    dists = np.concatenate(all_dists, axis=0)
    rho_full = float(dists.mean())

    if trim_quantile is None:
        rho_trim = rho_full
    else:
        q = np.quantile(dists, trim_quantile)
        mask = dists <= q
        if not np.any(mask):
            rho_trim = rho_full
        else:
            rho_trim = float(dists[mask].mean())

    return rho_full, rho_trim

# -----------------------------------
# Global Lipschitz L_γ in latent space (classifier from latent → logits)
# -----------------------------------
def estimate_L_gamma_feature_global() -> float:
    """
    Global Lipschitz constant of the margin in latent space:

      L_γ = sup_{y≠k} ||w_y - w_k||_2

    for the linear classifier M_f(z) = W z + b.
    """
    with torch.no_grad():
        W = model.classifier.weight.detach()   # [K, D_latent]
        diff = W.unsqueeze(0) - W.unsqueeze(1)  # [K, K, D]
        norms = diff.norm(p=2, dim=2)          # [K, K]
        Kc = norms.size(0)
        mask = ~torch.eye(Kc, dtype=torch.bool, device=norms.device)
        L_gamma = norms[mask].max().item()
    return float(L_gamma)

# -----------------------------------
# Helper: hinge loss on a loader under shift σ (oracle LB)
# -----------------------------------
@torch.no_grad()
def hinge_loss_on_loader(loader: DataLoader, sigma: float) -> float:
    """
    Compute L_h(f, loader shifted by σ).
    """
    model.eval()
    total_hinge = 0.0
    total_n = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        _, hinge = samplewise_margin_and_hinge(logits, y, margin=MARGIN)
        bs = x.size(0)
        total_hinge += hinge.sum().item()
        total_n += bs
    return total_hinge / max(total_n, 1)

# -----------------------------------
# Helper: margin drift δ_mix(σ)
# -----------------------------------
@torch.no_grad()
def estimate_margin_drift(
    loader: DataLoader,
    sigma: float,
    max_batches: int = None,
    trim_quantile: float = 0.9,
):
    """
    Estimate the average margin drift

        δ_mix(σ) ≈ E[ |γ_f(T_σ(X),Y) - γ_f(X,Y)| ],

    using labeled source data (X,Y) ~ P_{XY}.

    Returns (delta_full, delta_trimmed).
    """
    model.eval()
    all_diffs = []

    for b, (x, y) in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break

        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits_P = model.logits_from_x_nograd(x)
        margin_P, _ = samplewise_margin_and_hinge(logits_P, y, margin=MARGIN)

        x_shift = apply_Q_transform(x.clone(), sigma)
        logits_Q = model.logits_from_x_nograd(x_shift)
        margin_Q, _ = samplewise_margin_and_hinge(logits_Q, y, margin=MARGIN)

        diff = (margin_Q - margin_P).abs().detach().cpu().numpy()
        all_diffs.append(diff)

    if not all_diffs:
        return float("nan"), float("nan")

    diffs = np.concatenate(all_diffs, axis=0)
    delta_full = float(diffs.mean())

    if trim_quantile is None:
        delta_trim = delta_full
    else:
        q = np.quantile(diffs, trim_quantile)
        mask = diffs <= q
        if not np.any(mask):
            delta_trim = delta_full
        else:
            delta_trim = float(diffs[mask].mean())

    return delta_full, delta_trim

# -----------------------------------
# Score-space density sup M and empirical W1 (oracle)
# -----------------------------------
def estimate_density_sup(scores: np.ndarray, nbins: int = 50) -> float:
    """
    Crude estimate of sup score density from a histogram.
    """
    scores = np.asarray(scores, dtype=float)
    if scores.size == 0:
        return float("nan")
    lo, hi = scores.min(), scores.max()
    if not np.isfinite(lo) or not np.isfinite(hi):
        return float("nan")
    if hi == lo:
        return 1.0
    pad = 1e-6 * (hi - lo)
    lo -= pad
    hi += pad
    hist, edges = np.histogram(scores, bins=nbins, range=(lo, hi))
    bin_width = edges[1] - edges[0]
    pdf_est = hist / (scores.size * bin_width)
    return float(pdf_est.max())

def empirical_W1_1d(a: np.ndarray, b: np.ndarray) -> float:
    """
    Empirical W1 between two 1D samples with equal weights.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n = min(a.size, b.size)
    if n == 0:
        return float("nan")
    a_sorted = np.sort(a)[:n]
    b_sorted = np.sort(b)[:n]
    return float(np.mean(np.abs(a_sorted - b_sorted)))

# -----------------------------------
# 1) Coverage vs σ for empirical procedures
# -----------------------------------
sigma_vals  = SIGMA_GRID.astype(float)
nominal_cov = 1.0 - ALPHA

scores_P_cal_true = collect_true_scores(P_cal_loader, sigma=0.0)
q_source = conformal_quantile(scores_P_cal_true, ALPHA, finite_sample=True)

cov_source_Q = []
cov_hard_Q   = []
cov_alg_Q    = cov_Q_uP_epochs_sigma[-1, :].copy()  # source-tuned pseudo-cal at final epoch

print("Computing source and hard pseudo calibration coverage vs σ ...")

for sigma in sigma_vals:
    sigma_f = float(sigma)

    scores_Q_tst_true = collect_true_scores(Q_tst_loader, sigma=sigma_f)

    cov_src = coverage_given_threshold(scores_Q_tst_true, q_source)
    cov_source_Q.append(cov_src)

    scores_Q_cal_hard = collect_pseudo_scores_hard(
        loader=Q_cal_loader,
        sigma=sigma_f,
        num_classes=NUM_CLASSES,
    )
    q_hard = conformal_quantile(scores_Q_cal_hard, ALPHA, finite_sample=True)
    cov_hard = coverage_given_threshold(scores_Q_tst_true, q_hard)
    cov_hard_Q.append(cov_hard)

cov_source_Q = np.array(cov_source_Q, dtype=np.float64)
cov_hard_Q   = np.array(cov_hard_Q, dtype=np.float64)

# -----------------------------------
# 2) Source hinge loss L_h(f,P) at final epoch
# -----------------------------------
L_h_source = float(hinge_P_epochs[-1])
print(f"Final source hinge loss L_h(f,P) ≈ {L_h_source:.4f}")

# -----------------------------------
# 3) Margin drift δ_mix(σ)
# -----------------------------------
print("Estimating margin drift δ_mix(σ) on P (full & trimmed) ...")
delta_margin_vals = []
delta_margin_trim_vals = []

for sigma in sigma_vals:
    delta_full, delta_trim = estimate_margin_drift(
        loader=P_cal_loader,
        sigma=float(sigma),
        max_batches=None,
        trim_quantile=0.9,   # trimmed mean for tighter LB
    )
    delta_margin_vals.append(delta_full)
    delta_margin_trim_vals.append(delta_trim)

delta_margin_vals = np.array(delta_margin_vals, dtype=np.float64)
delta_margin_trim_vals = np.array(delta_margin_trim_vals, dtype=np.float64)
print("δ_mix(σ) full:   ", delta_margin_vals)
print("δ_mix(σ) trimmed:", delta_margin_trim_vals)

# -----------------------------------
# 4) Feature-space ρ_mix(σ) + global L_γ
# -----------------------------------
print("Estimating latent ρ_mix(σ) (full & trimmed) ...")
rho_feat_vals = []
rho_feat_trim_vals = []

for sigma in sigma_vals:
    rho_full, rho_trim = estimate_feature_rho_mix(
        loader=P_cal_loader,
        sigma=float(sigma),
        max_batches=None,
        trim_quantile=0.9,   # trimmed for tighter LB
    )
    rho_feat_vals.append(rho_full)
    rho_feat_trim_vals.append(rho_trim)

rho_feat_vals = np.array(rho_feat_vals, dtype=np.float64)
rho_feat_trim_vals = np.array(rho_feat_trim_vals, dtype=np.float64)
print("ρ_mix(σ) full:   ", rho_feat_vals)
print("ρ_mix(σ) trimmed:", rho_feat_trim_vals)

L_gamma_global = estimate_L_gamma_feature_global()
print(f"Global feature-space margin Lipschitz L_γ ≈ {L_gamma_global:.4f}")

# -----------------------------------
# 5) Oracle hinge loss L_h(f,Q_σ)
# -----------------------------------
print("Estimating hinge loss L_h(f,Q_σ) for each σ ...")
L_h_Q_sigmas = []
for sigma in sigma_vals:
    L_h_Q = hinge_loss_on_loader(Q_tst_loader, sigma=float(sigma))
    L_h_Q_sigmas.append(L_h_Q)
L_h_Q_sigmas = np.array(L_h_Q_sigmas, dtype=np.float64)
print("L_h(f,Q_σ):", L_h_Q_sigmas)

# -----------------------------------
# 6) Score-space W1 LB (oracle)
# -----------------------------------
cov_lb_W1 = []
M_vals    = []
W1_vals   = []

print("Estimating score-space W1 and density sup M(σ) on Q_cal (oracle) ...")
for sigma in sigma_vals:
    sigma_f = float(sigma)

    scores_Q_cal_true = collect_true_scores(Q_cal_loader, sigma=sigma_f)
    scores_Q_cal_pseudo = collect_pseudo_scores_hard(
        loader=Q_cal_loader,
        sigma=sigma_f,
        num_classes=NUM_CLASSES,
    )

    M_hat = estimate_density_sup(scores_Q_cal_pseudo, nbins=50)
    W1_emp = empirical_W1_1d(scores_Q_cal_pseudo, scores_Q_cal_true)

    M_vals.append(M_hat)
    W1_vals.append(W1_emp)

    gap = 2.0 * math.sqrt(max(0.0, M_hat * W1_emp))
    lb  = nominal_cov - gap
    lb  = max(0.0, min(1.0, lb))
    cov_lb_W1.append(lb)

cov_lb_W1 = np.array(cov_lb_W1, dtype=np.float64)
M_vals    = np.array(M_vals, dtype=np.float64)
W1_vals   = np.array(W1_vals, dtype=np.float64)

print("Estimated M(σ)  =", M_vals)
print("Estimated W1(σ) =", W1_vals)

# -----------------------------------
# 7) Build the four coverage LBs
# -----------------------------------
tau_const = 0.0  # τ = 0 as requested

# (i) Oracle LB: Cov ≥ 1−α − L_h(f,Q_σ)
cov_lb_oracle = nominal_cov - L_h_Q_sigmas
cov_lb_oracle = np.clip(cov_lb_oracle, 0.0, 1.0)

# (ii) Proposition Tight LB (margin drift, trimmed δ_mix, τ=0)
cov_lb_margin_tight = nominal_cov - (L_h_source + delta_margin_trim_vals)
cov_lb_margin_tight = np.clip(cov_lb_margin_tight, 0.0, 1.0)

# (iii) Proposition Tight LB (global Lγ, trimmed ρ_mix, τ=0)
cov_lb_rho_tight = nominal_cov - (L_h_source + L_gamma_global * rho_feat_trim_vals)
cov_lb_rho_tight = np.clip(cov_lb_rho_tight, 0.0, 1.0)

# (iv) Score-space W1 LB is cov_lb_W1 (already computed)

In [ ]:
import numpy as np
import torch

# -----------------------------
# Helper: hard pseudo-label scores on Q
# -----------------------------
@torch.no_grad()
def collect_pseudo_scores_hard(
    loader: DataLoader,
    sigma: float,
    num_classes: int = NUM_CLASSES,
) -> np.ndarray:
    """
    Hard pseudo-calibration:
      - apply T_sigma,
      - y_hat = argmax_k logits_k,
      - score s(x, y_hat) = max_{k != y_hat} logit_k - logit_{y_hat}.
    """
    model.eval()
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        yhat = logits.argmax(dim=1)
        s = scores_for_labels_from_logits(logits, yhat)
        all_scores.append(s)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])


# -----------------------------
# Build table for selected sigmas using final epoch
# -----------------------------
@torch.no_grad()
def make_cifar10_cov_ess_table_latex():
    # Use last epoch (EPOCHS) for the source-tuned stats
    epoch_for_table = EPOCHS          # 1-based
    eidx = epoch_for_table - 1        # 0-based

    # Choose a concise subset of shift levels for the table
    # (edit this list if you want other σ’s)
    TABLE_SIGMAS = [0.0, 0.2, 0.3, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.5]

    # Source calibration threshold: conformal on P_cal with true labels (σ=0)
    scores_P_cal_true = collect_true_scores(P_cal_loader, sigma=0.0)
    q_P_source = conformal_quantile(scores_P_cal_true, ALPHA, finite_sample=True)

    # Storage for table rows
    rows = []

    for sigma_req in TABLE_SIGMAS:
        # Use nearest σ in the pre-defined SIGMA_GRID for source-tuned stats
        s_idx = int(np.argmin(np.abs(SIGMA_GRID - sigma_req)))
        sigma_used = float(SIGMA_GRID[s_idx])

        # True scores on Q_tst at this sigma (shared across methods)
        scores_Q_tst_true = collect_true_scores(Q_tst_loader, sigma_used)

        # 1) Source calibration: threshold fixed from P_cal (σ=0), evaluate on Q_tst(σ)
        cov_source = coverage_given_threshold(scores_Q_tst_true, q_P_source)
        ess_source = expected_set_size_threshold(Q_tst_loader, sigma_used, q_P_source)

        # 2) Hard pseudo calibration on Q
        scores_Q_cal_pseudo_hard = collect_pseudo_scores_hard(
            loader=Q_cal_loader,
            sigma=sigma_used,
            num_classes=NUM_CLASSES,
        )
        q_Qhat_hard = conformal_quantile(scores_Q_cal_pseudo_hard, ALPHA, finite_sample=True)
        cov_hard = coverage_given_threshold(scores_Q_tst_true, q_Qhat_hard)
        ess_hard = expected_set_size_threshold(Q_tst_loader, sigma_used, q_Qhat_hard)

        # 3) Source-tuned pseudo calibration (u_P^*), already tracked during training
        cov_src_tuned = float(cov_Q_uP_epochs_sigma[eidx, s_idx])
        ess_src_tuned = float(ESS_Q_uP_epochs_sigma[eidx, s_idx])

        # 4) Target calibration (ideal): true-label conformal on Q_cal(σ), evaluate on Q_tst(σ)
        scores_Q_cal_true = collect_true_scores(Q_cal_loader, sigma_used)
        q_Q_true = conformal_quantile(scores_Q_cal_true, ALPHA, finite_sample=True)
        cov_target = coverage_given_threshold(scores_Q_tst_true, q_Q_true)
        ess_target = expected_set_size_threshold(Q_tst_loader, sigma_used, q_Q_true)

        rows.append({
            "sigma":    sigma_used,
            "cov_src":  cov_source,
            "ess_src":  ess_source,
            "cov_hard": cov_hard,
            "ess_hard": ess_hard,
            "cov_tuned": cov_src_tuned,
            "ess_tuned": ess_src_tuned,
            "cov_tgt":  cov_target,
            "ess_tgt":  ess_target,
        })

    # -----------------------------
    # Build LaTeX table string
    # -----------------------------
    nominal_cov = 1.0 - ALPHA

    header = r"""\begin{table}[t]
\centering
\caption{CIFAR10: Coverage and expected set size vs.\ shift level $\sigma$ (nominal $1-\alpha=%.2f$).}
\label{tab:cifar10_cov_ess}
\begin{tabular}{c|cc|cc|cc|cc}
\hline
 & \multicolumn{2}{c|}{Source cal} & \multicolumn{2}{c|}{Hard pseudo cal} & \multicolumn{2}{c|}{Source-tuned cal} & \multicolumn{2}{c}{Target cal (ideal)} \\
$\sigma$ & Cov & ESS & Cov & ESS & Cov & ESS & Cov & ESS \\
\hline
""" % nominal_cov

    body_lines = []
    for r in rows:
        line = (
            f"{r['sigma']:.2f} & "
            f"{100.0*r['cov_src']:.2f} & {r['ess_src']:.2f} & "
            f"{100.0*r['cov_hard']:.2f} & {r['ess_hard']:.2f} & "
            f"{100.0*r['cov_tuned']:.2f} & {r['ess_tuned']:.2f} & "
            f"{100.0*r['cov_tgt']:.2f} & {r['ess_tgt']:.2f} \\\\"
        )
        body_lines.append(line)

    footer = r"""
\hline
\end{tabular}
\end{table}
"""

    table_latex = header + "\n".join(body_lines) + footer
    print(table_latex)


# Run this after training is done to print the LaTeX table
make_cifar10_cov_ess_table_latex()


In [ ]:
save_path = "cifar10_linear_uP_model.pth"
torch.save(model.state_dict(), save_path)
print(f"Saved model weights to {save_path}")

In [ ]:
# ============================================================
# Coverage vs σ + ramp-loss & margin-tail lower bounds
#   Empirical coverage:
#       - Source calibration
#       - Hard pseudo-calibration
#       - Source-tuned pseudo-calibration (Alg 1)  [brown]
#       - ECP (entropy-scaled hard pseudo-cal)     [green]
#       - WQLCP (weighted q + recon scaling)       [purple]
#
#   Lower bounds (ramp + margins):
#       1) 1 − α − L_r(f, Q_σ)
#       2) 1 − α − ( L_r(f,P) + L_γ ρ_mix(σ) )
#       3) 1 − α − P( γ_f(X,Y) ≤ L_γ ρ_mix(σ) )
#       4) 1 − α − Q_σ( γ_f(X,Y) ≤ 0 )
# ============================================================

import math
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    import scienceplots
    plt.style.use(["science", "ieee", "no-latex"])
except ImportError:
    plt.style.use("default")

# ---------- Matplotlib style ----------
mpl.rcParams["font.size"] = 14
mpl.rcParams["axes.labelsize"] = 14
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["legend.fontsize"] = 9
mpl.rcParams["xtick.labelsize"] = 13
mpl.rcParams["ytick.labelsize"] = 13
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
plt.rcParams["lines.linewidth"] = 1.0

# --- Global color palette ---
COLOR_SOURCE       = "tab:red"
COLOR_HARD         = "tab:blue"
COLOR_SOURCE_TUNED = "tab:green"   # changed from green -> brown
COLOR_ECP          = "tab:brown"
COLOR_WQLCP        = "tab:purple"
COLOR_LB           = "k"

IEEE_WIDTH_1COL = 3.5  # inches
IEEE_ASPECT = 0.9
FIGSIZE_1COL = (IEEE_WIDTH_1COL, IEEE_WIDTH_1COL * IEEE_ASPECT)

SIGMA_MIN_GRID = 0.0
SIGMA_MAX_GRID = 2.0
N_SIGMA_GRID   = 11
SIGMA_GRID     = np.linspace(SIGMA_MIN_GRID, SIGMA_MAX_GRID, N_SIGMA_GRID)

EPS = 1e-12

# ============================================================
# Helper functions
# ============================================================

def coverage_given_threshold(scores: np.ndarray, q: float) -> float:
    """Empirical coverage P(s(X,Y) ≤ q)."""
    scores = np.asarray(scores, dtype=float)
    if scores.size == 0:
        return float("nan")
    return float(np.mean(scores <= q))

def quantile_higher_np(a: np.ndarray, q: float) -> float:
    a = np.asarray(a, dtype=np.float64)
    if a.size == 0:
        return float("nan")
    try:
        return float(np.quantile(a, q, method="higher"))
    except TypeError:
        return float(np.quantile(a, q, interpolation="higher"))

def inflate_threshold_monotone(q: float, w: float) -> float:
    """
    For rule s <= q with possibly-negative q:
      w>=1 should NEVER shrink sets.
      if q>=0: q_eff = q*w
      if q<0 : q_eff = q/w  (less negative => larger threshold)
    """
    w = float(max(1.0, w))
    q = float(q)
    return q * w if q >= 0.0 else q / w

def weighted_quantile_higher(values: np.ndarray, weights: np.ndarray, q: float) -> float:
    """
    Weighted right-continuous quantile:
      smallest v such that sum_{i: val_i<=v} w_i >= q * sum_i w_i
    """
    v = np.asarray(values, dtype=np.float64)
    w = np.asarray(weights, dtype=np.float64)
    assert v.shape[0] == w.shape[0]
    if v.size == 0:
        return float("nan")
    w = np.maximum(w, 0.0)
    sw = float(w.sum())
    if sw <= 0:
        return quantile_higher_np(v, q)

    order = np.argsort(v)
    v_sorted = v[order]
    w_sorted = w[order]
    cw = np.cumsum(w_sorted)
    target = q * sw
    idx = int(np.searchsorted(cw, target, side="left"))
    idx = max(0, min(idx, v_sorted.size - 1))
    return float(v_sorted[idx])

@torch.no_grad()
def collect_pseudo_scores_hard(
    loader: DataLoader,
    sigma: float,
    num_classes: int = NUM_CLASSES,
) -> np.ndarray:
    """
    Hard pseudo-cal scores on Q_σ:
        ŷ = argmax_k logit_k after T_σ;
        returns scores s(x, ŷ).
    """
    model.eval()
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        yhat = logits.argmax(dim=1)

        s = scores_for_labels_from_logits(logits, yhat)
        if isinstance(s, torch.Tensor):
            s_np = s.detach().cpu().numpy()
        else:
            s_np = np.asarray(s)
        all_scores.append(s_np)

    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

@torch.no_grad()
def collect_entropies(loader: DataLoader, sigma: float) -> np.ndarray:
    model.eval()
    ents = []
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        ent = predictive_entropy_from_logits(logits)
        ents.append(ent.detach().cpu().numpy().astype(np.float64))
    return np.concatenate(ents, axis=0) if ents else np.array([], dtype=np.float64)

@torch.no_grad()
def collect_recon_losses(loader: DataLoader, sigma: float) -> np.ndarray:
    """
    Per-sample recon loss L(x). Assumes your model is AE-like.
    Tries:
      - model(x) -> (logits, x_rec)
      - else: model.encode/decode
    """
    model.eval()
    losses = []
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)

        out = model(x)
        if isinstance(out, (tuple, list)) and len(out) >= 2:
            x_rec = out[1]
        elif hasattr(model, "encode") and hasattr(model, "decode"):
            z = model.encode(x)
            x_rec = model.decode(z)
        else:
            raise RuntimeError("WQLCP needs reconstructions. Model must return (logits, x_rec) or have encode/decode.")

        l = F.mse_loss(x_rec, x, reduction="none").view(x.size(0), -1).mean(dim=1)
        losses.append(l.detach().cpu().numpy().astype(np.float64))

    return np.concatenate(losses, axis=0) if losses else np.array([], dtype=np.float64)

@torch.no_grad()
def extract_features_batch(x: torch.Tensor) -> torch.Tensor:
    model.eval()
    return model.features_from_x_nograd(x)

@torch.no_grad()
def estimate_feature_rho_mix(
    loader: DataLoader,
    sigma: float,
    max_batches: int = None,
    trim_quantile: float = 0.9,
):
    model.eval()
    all_dists = []
    for b, (x, _) in enumerate(loader):
        if max_batches is not None and b >= max_batches:
            break
        x = x.to(DEVICE)
        z = extract_features_batch(x)
        x_shift = apply_Q_transform(x.clone(), sigma)
        z_shift = extract_features_batch(x_shift)
        d = (z - z_shift).norm(p=2, dim=1)
        all_dists.append(d.detach().cpu().numpy())

    if not all_dists:
        return float("nan"), float("nan")

    dists = np.concatenate(all_dists, axis=0)
    rho_full = float(dists.mean())

    if trim_quantile is None:
        rho_trim = rho_full
    else:
        q = np.quantile(dists, trim_quantile)
        mask = dists <= q
        rho_trim = float(dists[mask].mean()) if np.any(mask) else rho_full

    return rho_full, rho_trim

def estimate_L_gamma_feature_global() -> float:
    with torch.no_grad():
        W = model.classifier.weight.detach()
        diff = W.unsqueeze(0) - W.unsqueeze(1)
        norms = diff.norm(p=2, dim=2)
        Kc = norms.size(0)
        mask = ~torch.eye(Kc, dtype=torch.bool, device=norms.device)
        L_gamma = norms[mask].max().item()
    return float(L_gamma)

@torch.no_grad()
def estimate_ramp_loss_P(loader: DataLoader) -> float:
    model.eval()
    total_ramp = 0.0
    total_n = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        margin, _ = samplewise_margin_and_hinge(logits, y, margin=MARGIN)
        hinge = torch.relu(1.0 - margin)
        ramp = torch.clamp(hinge, max=1.0)
        bs = x.size(0)
        total_ramp += ramp.sum().item()
        total_n += bs
    return total_ramp / total_n if total_n > 0 else float("nan")

@torch.no_grad()
def estimate_ramp_loss_Q(loader: DataLoader, sigma: float) -> float:
    model.eval()
    total_ramp = 0.0
    total_n = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        margin, _ = samplewise_margin_and_hinge(logits, y, margin=MARGIN)
        hinge = torch.relu(1.0 - margin)
        ramp = torch.clamp(hinge, max=1.0)
        bs = x.size(0)
        total_ramp += ramp.sum().item()
        total_n += bs
    return total_ramp / total_n if total_n > 0 else float("nan")

@torch.no_grad()
def collect_margins_P(loader: DataLoader) -> np.ndarray:
    model.eval()
    all_margins = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        margin, _ = samplewise_margin_and_hinge(logits, y, margin=MARGIN)
        all_margins.append(margin.detach().cpu().numpy())
    return np.concatenate(all_margins, axis=0) if all_margins else np.array([])

@torch.no_grad()
def estimate_margin_tail_Q(loader: DataLoader, sigma: float, threshold: float = 0.0) -> float:
    model.eval()
    total_count = 0
    total_n = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        margin, _ = samplewise_margin_and_hinge(logits, y, margin=MARGIN)
        total_count += (margin <= threshold).sum().item()
        total_n += x.size(0)
    return total_count / total_n if total_n > 0 else float("nan")

# ============================================================
# Main computation
# ============================================================

sigma_vals = SIGMA_GRID.astype(float)
nominal_cov = 1.0 - ALPHA

print("1) Source calibration threshold at σ=0 ...")
scores_P_cal_true = collect_true_scores(P_cal_loader, sigma=0.0)
q_source = conformal_quantile(scores_P_cal_true, ALPHA, finite_sample=True)

print("2) Coverage curves for source, hard pseudo, source-tuned, ECP, WQLCP ...")

# Choose ECP entropy source ("Q_tst" = paper-like; "Q_cal" = no-peek)
ECP_U_FROM = "Q_tst"
ENT_LOADER = Q_tst_loader if ECP_U_FROM == "Q_tst" else Q_cal_loader
print(f"   ECP entropy source: {ECP_U_FROM}")

cov_source_Q = []
cov_hard_Q = []
cov_ecp_Q = []
cov_wqlcp_Q = []

for sigma in sigma_vals:
    sigma_f = float(sigma)

    # True scores on Q_tst under shift σ
    scores_Q_tst_true = collect_true_scores(Q_tst_loader, sigma=sigma_f)

    # Source calibration coverage using q_source
    cov_src = coverage_given_threshold(scores_Q_tst_true, q_source)
    cov_source_Q.append(cov_src)

    # Hard pseudo-calibration on Q_σ
    scores_Q_cal_hard = collect_pseudo_scores_hard(Q_cal_loader, sigma=sigma_f, num_classes=NUM_CLASSES)
    q_hard = conformal_quantile(scores_Q_cal_hard, ALPHA, finite_sample=True)
    cov_hard = coverage_given_threshold(scores_Q_tst_true, q_hard)
    cov_hard_Q.append(cov_hard)

    # ECP: inflate q_hard by entropy quantile factor
    ent = collect_entropies(ENT_LOADER, sigma=sigma_f)
    u_ent = quantile_higher_np(ent, 1.0 - float(ALPHA))
    w_ent = float(max(1.0, u_ent))
    q_ecp = inflate_threshold_monotone(q_hard, w_ent)
    cov_ecp = coverage_given_threshold(scores_Q_tst_true, q_ecp)
    cov_ecp_Q.append(cov_ecp)

    # WQLCP: weighted quantile on Q_cal + recon scaling factor from Q_tst recon-loss quantile
    L_test = collect_recon_losses(Q_tst_loader, sigma=sigma_f)
    RL_test = quantile_higher_np(L_test, 1.0 - float(ALPHA))
    g = float(max(1.0, RL_test))

    L_cal = collect_recon_losses(Q_cal_loader, sigma=sigma_f)
    w_cal = np.asarray(L_cal, dtype=np.float64) + EPS
    q_w = weighted_quantile_higher(scores_Q_cal_hard, w_cal, 1.0 - float(ALPHA))
    q_wqlcp = inflate_threshold_monotone(q_w, g)

    cov_wq = coverage_given_threshold(scores_Q_tst_true, q_wqlcp)
    cov_wqlcp_Q.append(cov_wq)

cov_source_Q = np.array(cov_source_Q, dtype=np.float64)
cov_hard_Q   = np.array(cov_hard_Q, dtype=np.float64)
cov_ecp_Q    = np.array(cov_ecp_Q, dtype=np.float64)
cov_wqlcp_Q  = np.array(cov_wqlcp_Q, dtype=np.float64)

# Source-tuned pseudo-calibration coverage (Algorithm 1), final epoch
cov_alg_Q = cov_Q_uP_epochs_sigma[-1, :].copy()

print("3) Source ramp loss L_r(f,P) ...")
L_r_source = estimate_ramp_loss_P(P_tst_loader)
print(f"   L_r(f,P) ≈ {L_r_source:.4f}")

print("4) Margins on P ...")
margins_P = collect_margins_P(P_tst_loader)
print(f"   Collected {margins_P.shape[0]} margins on P.")

print("5) Latent ρ_mix(σ) (full & trimmed) ...")
rho_feat_vals = []
rho_feat_trim_vals = []
for sigma in sigma_vals:
    rho_full, rho_trim = estimate_feature_rho_mix(
        loader=P_cal_loader,
        sigma=float(sigma),
        max_batches=None,
        trim_quantile=0.9,
    )
    rho_feat_vals.append(rho_full)
    rho_feat_trim_vals.append(rho_trim)

rho_feat_vals = np.array(rho_feat_vals, dtype=np.float64)
rho_feat_trim_vals = np.array(rho_feat_trim_vals, dtype=np.float64)

print("6) Global latent L_γ ...")
L_gamma_global = estimate_L_gamma_feature_global()
print(f"   L_γ (latent) ≈ {L_gamma_global:.4f}")

print("7) Ramp loss on Q_σ ...")
L_r_Q_sigmas = []
for sigma in sigma_vals:
    L_r_Q = estimate_ramp_loss_Q(Q_tst_loader, sigma=float(sigma))
    L_r_Q_sigmas.append(L_r_Q)
L_r_Q_sigmas = np.array(L_r_Q_sigmas, dtype=np.float64)

print("8) Oracle margin tail Q_σ(γ_f(X,Y) ≤ 0) ...")
margin_tail_Q_sigmas = []
for sigma in sigma_vals:
    tail = estimate_margin_tail_Q(Q_tst_loader, sigma=float(sigma), threshold=0.0)
    margin_tail_Q_sigmas.append(tail)
margin_tail_Q_sigmas = np.array(margin_tail_Q_sigmas, dtype=np.float64)

# ============================================================
# Build ramp-loss-based & margin-based lower bounds
# ============================================================

print("9) Building coverage lower bounds...")

rho_mix_vals = rho_feat_trim_vals

cov_lb_ramp_Q = np.clip(nominal_cov - L_r_Q_sigmas, 0.0, 1.0)
cov_lb_ramp_Prho = np.clip(nominal_cov - (L_r_source + L_gamma_global * rho_mix_vals), 0.0, 1.0)

cov_lb_margin_tail_P = []
for rho in rho_mix_vals:
    theta = L_gamma_global * rho
    tail_prob = float(np.mean(margins_P <= theta)) if margins_P.size and np.isfinite(theta) else 0.0
    cov_lb_margin_tail_P.append(np.clip(nominal_cov - tail_prob, 0.0, 1.0))
cov_lb_margin_tail_P = np.array(cov_lb_margin_tail_P, dtype=np.float64)

cov_lb_margin0_Q = np.clip(nominal_cov - margin_tail_Q_sigmas, 0.0, 1.0)

# ============================================================
# Final plot
# ============================================================

fig, ax = plt.subplots(figsize=FIGSIZE_1COL)

# Empirical coverage
ax.plot(sigma_vals, cov_source_Q, color=COLOR_SOURCE, linestyle="-", label="Source Cal.")
ax.plot(sigma_vals, cov_hard_Q,   color=COLOR_HARD,   linestyle="-", label="Hard Pseudo-Cal.")
ax.plot(sigma_vals, cov_alg_Q,    color=COLOR_SOURCE_TUNED, linestyle="-", label="Source-tuned Pseudo-Cal.")
ax.plot(sigma_vals, cov_ecp_Q,    color=COLOR_ECP,    linestyle="-", label="ECP")
ax.plot(sigma_vals, cov_wqlcp_Q,  color=COLOR_WQLCP,  linestyle="-", label="WQLCP")

# Lower bounds
ax.plot(sigma_vals, cov_lb_ramp_Q,    color=COLOR_LB, linestyle="--", label=r"$1-\alpha - L_r(f,Q_\sigma)$")
ax.plot(sigma_vals, cov_lb_ramp_Prho, color=COLOR_LB, linestyle=":",  label=r"$1-\alpha - (L_r(f,P)+L_\gamma\rho_{\mathrm{mix}}(\sigma))$")

# Nominal
ax.axhline(nominal_cov, linestyle="--", color="gray", label=r"Nominal $1-\alpha$")

ax.set_xlabel(r"Shift level $\sigma$")
ax.set_ylabel(r"Coverage on $Q_\sigma$")
ax.set_ylim(0.0, 1.02)
ax.set_xlim(sigma_vals.min(), 1.5)
ax.grid(True, which="both", linewidth=0.5, alpha=0.5)
ax.legend(frameon=False, loc="lower left")

plt.tight_layout()
plt.savefig("cifar10_coverage_vs_shift_ramp_margin_bounds_plus_ecp_wqlcp.pdf")
plt.savefig("cifar10_coverage_vs_shift_ramp_margin_bounds_plus_ecp_wqlcp.png", dpi=300)
plt.show()


In [ ]:
# ============================================================
# Tau-adjusted hard pseudo-calibration using HINGE-loss bound
# + ORACLE (label-based) conformal on Q_sigma for comparison
#
# De-crowded plot (IEEE-friendly):
#   - Legend shows ONLY color->method using marker-only handles (colored circles)
#   - Linestyle meaning (solid/dashed/dash-dot) should be explained in the caption
#
# Requires (already defined in your environment):
#   model, DEVICE, MARGIN, NUM_CLASSES, ALPHA, SIGMA_GRID
#   Q_cal_loader, Q_tst_loader, P_cal_loader, P_tst_loader
#   apply_Q_transform, samplewise_margin_and_hinge
#   scores_for_labels_from_logits, conformal_quantile
#   collect_true_scores, coverage_given_threshold, expected_set_size_threshold
# ============================================================

import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D

try:
    import scienceplots
    plt.style.use(["science", "ieee", "no-latex"])
except ImportError:
    plt.style.use("default")

# --- Matplotlib style (IEEE-ish) ---
mpl.rcParams["font.size"] = 14
mpl.rcParams["axes.labelsize"] = 14
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["legend.fontsize"] = 9
mpl.rcParams["xtick.labelsize"] = 13
mpl.rcParams["ytick.labelsize"] = 13
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
plt.rcParams["lines.linewidth"] = 1.0

color_cov_base   = "tab:blue"
color_cov_tau    = "tab:orange"
color_cov_oracle = "tab:cyan"
color_LB         = "k"

IEEE_WIDTH_1COL = 3.5
IEEE_ASPECT     = 0.9
FIGSIZE_1COL    = (IEEE_WIDTH_1COL, IEEE_WIDTH_1COL * IEEE_ASPECT)

DATASET_TAG = "cifar10"

# ------------------------------------------------------------
# Helper: collect margins (using labeled data)
# ------------------------------------------------------------
@torch.no_grad()
def collect_margins(loader: DataLoader, sigma: float = 0.0) -> np.ndarray:
    model.eval()
    all_margins = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if float(sigma) != 0.0:
            x = apply_Q_transform(x, float(sigma))
        logits = model.logits_from_x_nograd(x)
        margin, _ = samplewise_margin_and_hinge(logits, y, margin=MARGIN)
        all_margins.append(margin.detach().cpu().numpy())
    return np.concatenate(all_margins, axis=0) if all_margins else np.array([])

def hinge_loss_from_margins(margins: np.ndarray) -> float:
    if margins.size == 0:
        return 0.0
    return float(np.mean(np.maximum(1.0 - margins, 0.0)))

# ------------------------------------------------------------
# Helper: hard pseudo-calibration scores (pseudo-label = argmax logit)
# ------------------------------------------------------------
@torch.no_grad()
def collect_pseudo_scores_hard(loader: DataLoader, sigma: float) -> np.ndarray:
    model.eval()
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        if float(sigma) != 0.0:
            x = apply_Q_transform(x, float(sigma))
        logits = model.logits_from_x_nograd(x)
        yhat = logits.argmax(dim=1)
        s = scores_for_labels_from_logits(logits, yhat)
        s_np = s.detach().cpu().numpy() if isinstance(s, torch.Tensor) else np.asarray(s)
        all_scores.append(s_np)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

# ============================================================
# STEP 1: Compute Delta_P on SOURCE P (can be negative)
# ============================================================

nominal_cov = 1.0 - ALPHA

print("\n=== STEP 1: Compute Delta_P on source P (NO clamping) ===")

scores_P_cal_pseudo = collect_pseudo_scores_hard(P_cal_loader, sigma=0.0)
q_pseudo_P = conformal_quantile(scores_P_cal_pseudo, ALPHA, finite_sample=True)

scores_P_tst_true = collect_true_scores(P_tst_loader, sigma=0.0)
coverage_P_pseudo = coverage_given_threshold(scores_P_tst_true, q_pseudo_P)

Delta_P = float(nominal_cov - coverage_P_pseudo)

margins_P = collect_margins(P_tst_loader, sigma=0.0)
L_h_P = hinge_loss_from_margins(margins_P)

denom_base = float(L_h_P - Delta_P)
target_LB  = float(nominal_cov - L_h_P + Delta_P)

print(f"Nominal coverage (1-α):                 {nominal_cov:.6f}")
print(f"Pseudo-cal threshold on P (q_pseudo_P): {q_pseudo_P:.6f}")
print(f"Coverage on P using q_pseudo_P:         {coverage_P_pseudo:.6f}")
print(f"Delta_P = (1-α) - coverage:             {Delta_P:.6f}   (can be negative)")
print(f"Hinge loss L_h(P):                      {L_h_P:.6f}")
print(f"Denominator L_h(P) - Delta_P:           {denom_base:.6f}")
print(f"Target LB = (1-α) - L_h(P) + Delta_P:   {target_LB:.6f}")

# ============================================================
# STEP 2: Sweep sigma, compute hard / tau / ORACLE
# ============================================================

sigma_vals = SIGMA_GRID.astype(float)
N_SIGMA    = len(sigma_vals)

coverage_hard       = np.zeros(N_SIGMA, dtype=np.float64)
coverage_hard_tau   = np.zeros(N_SIGMA, dtype=np.float64)
coverage_oracle     = np.zeros(N_SIGMA, dtype=np.float64)

ESS_hard            = np.zeros(N_SIGMA, dtype=np.float64)
ESS_hard_tau        = np.zeros(N_SIGMA, dtype=np.float64)
ESS_oracle          = np.zeros(N_SIGMA, dtype=np.float64)

tau_by_sigma        = np.zeros(N_SIGMA, dtype=np.float64)
LB_empirical_sigma  = np.zeros(N_SIGMA, dtype=np.float64)
hinge_loss_Q_sigma  = np.zeros(N_SIGMA, dtype=np.float64)

print("\n=== STEP 2: Sweep sigma, solve tau(sigma), and compute oracle q_Q(σ) ===")

TAU_MIN = 0.0
TAU_MAX = None
EPS = 1e-12

for i, sigma in enumerate(sigma_vals):
    sigma_f = float(sigma)
    print(f"\nσ = {sigma_f:.3f}")

    # --- A) Hard pseudo-calibration on Q_sigma: q_hard(σ) ---
    scores_Q_cal_hard = collect_pseudo_scores_hard(Q_cal_loader, sigma=sigma_f)
    q_hard = conformal_quantile(scores_Q_cal_hard, ALPHA, finite_sample=True)

    # --- B) Oracle conformal on Q_sigma (labels): q_oracle(σ) ---
    scores_Q_cal_true = collect_true_scores(Q_cal_loader, sigma=sigma_f)
    q_oracle = conformal_quantile(scores_Q_cal_true, ALPHA, finite_sample=True)

    # --- True scores on Q_tst ---
    scores_Q_tst_true = collect_true_scores(Q_tst_loader, sigma=sigma_f)

    # Hard baseline
    cov_base = coverage_given_threshold(scores_Q_tst_true, q_hard)
    ESS_base = expected_set_size_threshold(Q_tst_loader, sigma_f, q_hard)
    coverage_hard[i] = cov_base
    ESS_hard[i]      = ESS_base

    # Oracle baseline
    cov_or = coverage_given_threshold(scores_Q_tst_true, q_oracle)
    ESS_or = expected_set_size_threshold(Q_tst_loader, sigma_f, q_oracle)
    coverage_oracle[i] = cov_or
    ESS_oracle[i]      = ESS_or

    # --- C) τ-adjustment using labeled Q_tst hinge loss (diagnostic) ---
    margins_Q = collect_margins(Q_tst_loader, sigma=sigma_f)
    L_h_Q = hinge_loss_from_margins(margins_Q)
    hinge_loss_Q_sigma[i] = L_h_Q

    if abs(denom_base) <= EPS:
        tau_raw = 0.0
        if L_h_Q > EPS:
            print("  WARNING: L_h(P)-Δ_P is ~0 but L_h(Qσ)>0; equation not solvable with finite τ.")
    else:
        tau_raw = 2.0 * (L_h_Q / denom_base - 1.0)

    tau = float(max(TAU_MIN, tau_raw))
    if TAU_MAX is not None:
        tau = float(np.clip(tau, TAU_MIN, TAU_MAX))
    tau_by_sigma[i] = tau

    denom = 1.0 + tau / 2.0
    LB_empirical_sigma[i] = float(nominal_cov - (L_h_Q / denom)) if denom > 0 else float("nan")

    q_hard_tau = q_hard + tau
    cov_tau = coverage_given_threshold(scores_Q_tst_true, q_hard_tau)
    ESS_tau = expected_set_size_threshold(Q_tst_loader, sigma_f, q_hard_tau)
    coverage_hard_tau[i] = cov_tau
    ESS_hard_tau[i]      = ESS_tau

    print(f"  q_hard(σ)                = {q_hard:.6f}")
    print(f"  q_oracle(σ)              = {q_oracle:.6f}")
    print(f"  L_h(Q_σ)                 = {L_h_Q:.6f}")
    print(f"  τ_raw(σ)                 = {tau_raw:.6f}")
    print(f"  τ(σ)=max(0,τ_raw)        = {tau:.6f}")
    print(f"  LB_tau(σ)                = {LB_empirical_sigma[i]:.6f}")
    print(f"  Coverage: hard={cov_base:.4f}, tau={cov_tau:.4f}, oracle={cov_or:.4f}")
    print(f"  ESS     : hard={ESS_base:.2f}, tau={ESS_tau:.2f}, oracle={ESS_or:.2f}")

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"Delta_P: {Delta_P:.6f}  (no clamping)")
print(f"L_h(P):  {L_h_P:.6f}")
print(f"L_h(P)-Delta_P: {denom_base:.6f}")
print(f"Target LB: {target_LB:.6f}")

# ============================================================
# Plot: coverage + expected set size vs sigma (DE-CROWDED)
# ============================================================

fig, ax1 = plt.subplots(figsize=FIGSIZE_1COL)
ax2 = ax1.twinx()

# Coverage (solid)
ax1.plot(sigma_vals, coverage_hard,     "-",  color=color_cov_base)
ax1.plot(sigma_vals, coverage_hard_tau, "-",  color=color_cov_tau)
ax1.plot(sigma_vals, coverage_oracle,   "-",  color=color_cov_oracle)

# Hinge-loss LB (dash-dot, black)
ax1.plot(sigma_vals, LB_empirical_sigma, "--", color=color_LB)

# ESS (dashed)
ax2.plot(sigma_vals, ESS_hard,     ":", color=color_cov_base)
ax2.plot(sigma_vals, ESS_hard_tau, ":", color=color_cov_tau)
ax2.plot(sigma_vals, ESS_oracle,   ":", color=color_cov_oracle)

ax1.set_xlabel(r"Shift level $\sigma$")
ax1.set_ylabel(r"Coverage on $Q_\sigma$")
ax1.set_ylim(0.0, 1.02)
ax1.set_xlim(float(sigma_vals.min()), 1.5)
ax1.grid(True, which="both", linewidth=0.5, alpha=0.5)

ax2.set_ylabel(r"Expected set size on $Q_\sigma$")
ax2.set_ylim(0.0, NUM_CLASSES + 0.5)

# --- Legend: color -> method (marker-only circles) ---
method_handles = [
    Line2D([0], [0], marker='o', linestyle='None', markersize=6,
           markerfacecolor=color_cov_base, markeredgecolor=color_cov_base),
    Line2D([0], [0], marker='o', linestyle='None', markersize=6,
           markerfacecolor=color_cov_tau, markeredgecolor=color_cov_tau),
    Line2D([0], [0], marker='o', linestyle='None', markersize=6,
           markerfacecolor=color_cov_oracle, markeredgecolor=color_cov_oracle),
]
method_labels = [
    r"$q_{\tilde{Q}_\sigma,\alpha}$",
    r"$q_{\tilde{Q}_\sigma,\alpha}+\tau(\sigma)$",
    r"$q_{Q_\sigma,\alpha}$",
]
# ax1.legend(
#     method_handles, method_labels,
# #     title="Method (color)",
#     frameon=False,
#     loc="lower left", bbox_to_anchor=(0.01, 0.1),
#     handletextpad=0.6,
#     borderaxespad=0.0
# )

plt.tight_layout()
plt.savefig(f"{DATASET_TAG}_coverage_ess_vs_shift_pseudocal_tau_plus_oracle.pdf")
plt.savefig(f"{DATASET_TAG}_coverage_ess_vs_shift_pseudocal_tau_plus_oracle.png", dpi=300)
plt.show()

print(f"\nSaved plot: {DATASET_TAG}_coverage_ess_vs_shift_pseudocal_tau_plus_oracle.[pdf|png]")
